# 4 - Sentiment: which requesters are frustrated?

**Question.** Ticket text and status updates carry tone. Which analyser finds
the frustrated requester early enough to do something about it?

| Approach | What it is |
|---|---|
| `lexicon` | FR/EN frustration markers, negation, intensifiers, shouting (shipped) |
| `transformer` | Multilingual sentiment checkpoint |
| `llm` | Claude, constrained to the three labels |

**Every dependency is mandatory** - the transformer and LLM arms used to be
skippable extras. The only thing an install cannot provide is API credentials
and a budget, so `RUN_LLM` is a *spend* gate, not a capability gate.

### The metric is not accuracy, and the reason is asymmetric cost

This is a **triage signal**, not an analytics number. A missed frustrated
requester is an escalation, a complaint, a churned internal customer; a false
positive is thirty seconds of an agent's attention. So the headline is
**recall on `negative`** at a precision the desk can live with, with precision
reported beside it rather than one optimised alone.

**Kappa, not accuracy**, for the aggregate: the class balance is heavily
neutral, so accuracy rewards an analyser for saying neutral to everything.

### The honest problem with this notebook

**There are no human sentiment labels.** The reference is a weak label derived
from signals correlated with frustration - reopen count, escalation, breached
SLA, reassignment churn - not from anyone reading the ticket. Every number here
measures *agreement with that heuristic*, and an analyser can only look good by
resembling it. Section 5 is where that limit gets tested rather than ignored.

```bash
uv sync
uv run itsm data build
```

### Runtime

The transformer arm batches (`batch_size` on its pipeline) and its checkpoint
is cached process-wide. The LLM arm is network-bound, not compute-bound, so it
is issued from a thread pool - 8 in flight turns 200 sequential round trips of
about a second each into well under a minute.

In [ ]:
import json
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)

from itsm.config import get_settings
from itsm.models.sentiment import build_sentiment
from itsm.pipelines.anonymise import load_raw_tickets
from itsm.pipelines.eval_sentiment import operational_weak_labels

pd.set_option('display.width', 170)
pd.set_option('display.max_columns', 40)

settings = get_settings()
COLUMN = 'description'
SAMPLE = 800
SEED = 42
LABELS = ['negative', 'neutral', 'positive']

RUN_LLM = False       # spend gate; the package is installed, the budget is yours
LLM_CAP = 200
LLM_MODEL = 'claude-opus-5'
LLM_WORKERS = 8       # concurrent requests; the work is IO-bound

## 1. The sample and the weak label

Read the class balance before anything else: if `negative` is 8% of the sample,
an analyser that never predicts it scores 92% accuracy.

In [ ]:
tickets = load_raw_tickets(settings)
sample = tickets.sample(n=min(SAMPLE, len(tickets)), random_state=SEED).reset_index(drop=True)
texts = sample[COLUMN].fillna('').astype(str).tolist()
gold = list(operational_weak_labels(sample))

print(len(sample), 'tickets')
balance = pd.Series(gold).value_counts().to_frame('weak label')
balance['share'] = (balance['weak label'] / len(gold)).round(4)
balance

## 2. Run the analysers

In [ ]:
EXPECTED = {
    'lexicon': 'high recall on negative, moderate precision; transparent evidence per ticket',
    'transformer': 'better on implicit frustration with no marker word; slower, opaque',
    'llm': 'best on sarcasm and politeness-masked anger; the only one with a bill',
}

predictions, timings, detailed = {}, {}, {}

for backend in ('lexicon', 'transformer'):
    analyser = build_sentiment(backend, model_name=settings.sentiment_model)
    analyser.analyze(['warm up'])   # load the checkpoint outside the clock
    started = time.perf_counter()
    results = analyser.analyze(texts)
    timings[backend] = time.perf_counter() - started

    predictions[backend] = [result.label for result in results]
    detailed[backend] = results   # kept for the evidence table in section 5
    print(f'{backend:12s} {timings[backend]:6.1f}s'
          f'  {1000 * timings[backend] / len(texts):7.2f} ms/ticket')

In [ ]:
import anthropic

SYSTEM = (
    'You read IT service-desk tickets and judge the requester emotional tone.\n\n'
    'negative: frustrated, angry, blocked, complaining, escalating, or resigned.\n'
    'neutral: a factual report with no emotional content. Most tickets are neutral.\n'
    'positive: thanks, praise, or relief.\n\n'
    'Tickets are in French or English and often terse. Urgency is not frustration: '
    'a blunt "serveur down, urgent" is a neutral factual report. Politeness is not '
    'positivity. Judge the requester tone, not the severity of the incident.'
)

SCHEMA = {
    'type': 'json_schema',
    'schema': {
        'type': 'object',
        'properties': {'label': {'type': 'string', 'enum': LABELS}},
        'required': ['label'],
        'additionalProperties': False,
    },
}

PRICING = {'claude-opus-5': (5.0, 25.0), 'claude-sonnet-5': (2.0, 10.0),
           'claude-haiku-4-5': (1.0, 5.0)}

client = anthropic.Anthropic()
llm_cost = {}


def classify_one(text):
    '''One request. Returns (label, input_tokens, output_tokens, failed).'''
    try:
        response = client.messages.create(
            model=LLM_MODEL,
            max_tokens=64,
            system=[{'type': 'text', 'text': SYSTEM, 'cache_control': {'type': 'ephemeral'}}],
            output_config={'format': SCHEMA, 'effort': 'low'},
            messages=[{'role': 'user', 'content': 'Ticket:\n' + text}],
        )
        payload = json.loads(next(b.text for b in response.content if b.type == 'text'))
        return payload['label'], response.usage.input_tokens, response.usage.output_tokens, 0
    except anthropic.APIError:
        # Counted, not hidden: one failure must not lose the run, and the
        # fallback keeps the predictions aligned with the gold labels.
        return 'neutral', 0, 0, 1


if RUN_LLM:
    subset_texts = texts[:LLM_CAP]
    started = time.perf_counter()
    # Network-bound, so concurrency is the whole optimisation. The SDK retries
    # 429s on its own, so a pool this size does not need its own backoff.
    with ThreadPoolExecutor(max_workers=LLM_WORKERS) as pool:
        answers = list(pool.map(classify_one, subset_texts))
    elapsed = time.perf_counter() - started

    predictions['llm'] = [a[0] for a in answers]
    timings['llm'] = elapsed
    tokens_in = sum(a[1] for a in answers)
    tokens_out = sum(a[2] for a in answers)
    rate_in, rate_out = PRICING[LLM_MODEL]
    cost = (tokens_in * rate_in + tokens_out * rate_out) / 1e6
    llm_cost = {
        'USD (this run)': round(cost, 4),
        'USD per 1k tickets': round(1000 * cost / len(subset_texts), 4),
        'failed calls': sum(a[3] for a in answers),
    }
    print(f'llm {elapsed:.1f}s for {len(subset_texts)} tickets ->', llm_cost)
else:
    print('LLM arm not run - set RUN_LLM = True in cell 2.')

## 3. Headline

**`negative` recall is the row that decides**, with its precision next to it.
Missing a frustrated requester costs an escalation; a false positive costs an
agent thirty seconds.

Kappa is the aggregate to trust - it discounts the agreement you would get by
chance from a heavily neutral distribution, which accuracy does not.

In [ ]:
def score(y_true, y_pred, seconds, n):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=LABELS, zero_division=0
    )
    index = LABELS.index('negative')
    return {
        'negative recall': round(float(recall[index]), 4),
        'negative precision': round(float(precision[index]), 4),
        'negative F1': round(float(f1[index]), 4),
        'macro F1': round(float(f1_score(y_true, y_pred, labels=LABELS, average='macro',
                                         zero_division=0)), 4),
        'kappa': round(float(cohen_kappa_score(y_true, y_pred, labels=LABELS)), 4),
        'accuracy': round(float(accuracy_score(y_true, y_pred)), 4),
        'ms per ticket': round(1000 * seconds / n, 2),
    }


scores = {}
for name, predicted in predictions.items():
    scores[name] = score(gold[:len(predicted)], predicted, timings[name], len(predicted))
    scores[name]['n scored'] = len(predicted)
    scores[name]['expected'] = EXPECTED[name]
if llm_cost:
    scores['llm'].update(llm_cost)

comparison = pd.DataFrame(scores)
comparison

## 4. Where the disagreement is

The confusion matrix answers the question the headline cannot: when an analyser
is wrong about `negative`, does it call it `neutral` (a miss - costly) or does
it call neutral tickets `negative` (noise - cheap)?

In [ ]:
for name, predicted in predictions.items():
    matrix = pd.DataFrame(
        confusion_matrix(gold[:len(predicted)], predicted, labels=LABELS, normalize='true'),
        index=['true ' + label for label in LABELS],
        columns=['pred ' + label for label in LABELS],
    )
    print('===', name, '===')
    display(matrix.round(3))

## 5. The disagreements are the actual experiment

Because the reference is a **weak label and not a human judgement**, a
disagreement is not automatically an analyser error. The tickets below are
where the analysers and the heuristic diverge, printed so a person can read
twenty of them and decide who was right.

**That reading is the only thing in this notebook that can tell you whether the
weak label is any good.** Every metric above inherits it.

In [ ]:
primary = 'lexicon'
predicted = predictions[primary]

rows = []
for position, (truth, pred) in enumerate(zip(gold[:len(predicted)], predicted)):
    if truth == pred:
        continue
    row = {'weak label': truth, primary: pred, 'text': texts[position][:160]}
    for other in predictions:
        if other != primary and position < len(predictions[other]):
            row[other] = predictions[other][position]
    rows.append(row)

print(len(rows), 'disagreements out of', len(predicted))
pd.DataFrame(rows).head(20)

In [ ]:
# The lexicon is the only analyser that can show its work. For a signal that
# may route a ticket to a human, being able to say *why* is worth real points
# against a slightly better opaque score.
for result, text in list(zip(detailed['lexicon'], texts))[:200]:
    if result.label == 'negative' and result.evidence:
        print(round(result.score, 2), result.evidence, '->', text[:110])

## 6. Conclusion

*Fill in from the run. The structure of the argument:*

**What ships.** The analyser with the best `negative` recall at a precision the
desk will tolerate - and if the lexicon is close, it wins on two grounds the
table does not score: it runs offline at microseconds per ticket, and it can
show the markers that produced the verdict. A frustration flag an agent cannot
interrogate is a flag they will learn to ignore.

**The ceiling is the label, not the model.** Every score here is agreement with
a heuristic built from reopens, escalations and SLA breaches. An analyser that
scores badly may be reading the ticket correctly while the heuristic is wrong,
and section 5 is where that is checked by hand.

**Before this drives anything user-visible**, annotate 300 tickets by hand,
with two annotators and an inter-annotator agreement number. Sentiment is
subjective enough that a kappa between two humans is the real ceiling, and
without it there is no way to know whether 0.66 is good.

**Threats to validity:**

* Weak labels, as above.
* Generated text. Real frustration is sarcastic, passive-aggressive, and in
  French half the time; no generator reproduces that.
* The LLM arm ran on `LLM_CAP` tickets - a different sample size to the others.
  The table reports `n scored` so the comparison is not read as like-for-like
  when it is not.

In [ ]:
settings.ensure_dirs()
path = settings.reports_dir / 'experiment_04_sentiment.json'
path.write_text(
    json.dumps(
        {
            'column': COLUMN,
            'n_tickets': len(sample),
            'label_source': 'weak labels from reopens / escalation / SLA breach',
            'class_balance': balance.to_dict(),
            'llm_ran': RUN_LLM,
            'headline': comparison.to_dict(),
            'n_disagreements': len(rows),
        },
        indent=2,
        default=str,
    ),
    encoding='utf-8',
)
print('written to', path)